<!--
Copyright 2026 Yaroslav Mariukha
SPDX-License-Identifier: Apache-2.0
-->


# Avalon-ST Bus Helpers

These cells are meant to be copied into a cocotb test. They need a simulator DUT with Avalon-ST ports.

In [ ]:
import cocotb
from cocotb.clock import Clock
from cocotb.triggers import RisingEdge

from fpga_verification.sim.buses import (
    AvalonFormat,
    AvalonSTBus,
    AvalonSTFrame,
    AvalonSTSink,
    AvalonSTSource,
)


def pause_every_fourth_cycle():
    while True:
        yield False
        yield False
        yield False
        yield True


@cocotb.test()
async def stream_loopback_test(dut):
    cocotb.start_soon(Clock(dut.clk, 10, units="ns").start())

    dut.reset.value = 1
    await RisingEdge(dut.clk)
    dut.reset.value = 0

    fmt = AvalonFormat(
        bits_per_symbol=8,
        symbols_per_beat=4,
        first_symbol_in_high_order_bits=False,
    )

    source = AvalonSTSource(
        AvalonSTBus.from_prefix(dut, "sink"),
        fmt,
        dut.clk,
        reset=dut.reset,
        packets=True,
        idle_value=0,
    )
    sink = AvalonSTSink(
        AvalonSTBus.from_prefix(dut, "source"),
        fmt,
        dut.clk,
        reset=dut.reset,
        packets=True,
    )
    sink.set_pause_generator(pause_every_fourth_cycle())

    await source.send(AvalonSTFrame([0x11, 0x22, 0x33, 0x44, 0x55]))
    received = await sink.recv()

    assert received.data == [0x11, 0x22, 0x33, 0x44, 0x55]

`AvalonSTMonitor` is passive; use it when the testbench must observe an existing valid/ready stream without driving ready.

In [ ]:
from fpga_verification.sim.buses import AvalonSTMonitor

# monitor = AvalonSTMonitor(AvalonSTBus.from_prefix(dut, "tap"), fmt, dut.clk,
#                           reset=dut.reset, packets=True)
# frame = await monitor.recv()
# beat = await monitor.recv_beat()